In [ ]:
from IPython.display import display
import json
import time
import joblib
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
import optuna
import pandas as pd
from scipy.stats import pearsonr, wilcoxon
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, cross_val_score
from xgboost import XGBRegressor

In [ ]:
#  wgranie danych które przeszły już przez klastrowanie
X_train = pd.read_pickle('X_train_with_clusters.pkl')
X_test = pd.read_pickle('X_test_with_clusters.pkl')
y_train = pd.read_pickle('y_train.pkl')
y_test = pd.read_pickle('y_test.pkl')

# Oczyszczanie ze zmiennych tekstowych
kolumny_do_usuniecia = ['brand_name', 'model_name']
X_train_final = X_train.drop(columns=[col for col in kolumny_do_usuniecia if col in X_train.columns])
X_test_final = X_test.drop(columns=[col for col in kolumny_do_usuniecia if col in X_test.columns])
X_test_final = X_test_final.reindex(columns=X_train_final.columns, fill_value=0)


In [ ]:
# inicjalizacja modeli bazowych
modele = {
    'Random Forest (Baseline)': RandomForestRegressor(random_state=42, n_jobs=-1),
    'XGBoost (Baseline)': XGBRegressor(random_state=42, n_jobs=-1)
}

wyniki_lista = []
for nazwa, model in modele.items():
    model.fit(X_train_final, y_train)
    y_pred = model.predict(X_test_final)
    wyniki_lista.append({
        'Model': nazwa,
        'MAE (PLN)': round(mean_absolute_error(y_test, y_pred), 2),
        'RMSE (PLN)': round(np.sqrt(mean_squared_error(y_test, y_pred)), 2),
        'R2 Score': round(r2_score(y_test, y_pred), 4)
    })
print("MODELI BAZOWE OCENIONE")
display(pd.DataFrame(wyniki_lista))


In [ ]:
optuna.logging.set_verbosity(optuna.logging.WARNING)



print("ROZPOCZYNANIE OPTYMALIZACJI BAYESOWSKIEJ (OPTUNA)")

kf = KFold(n_splits=5, shuffle=True, random_state=42)



# funkcja celu dla Random Forest

def objective_rf(trial):

    # zdefiniowana przestrzeń poszukiwań parametrów

    params = {

        'n_estimators': trial.suggest_int('n_estimators', 100, 500),

        'max_depth': trial.suggest_int('max_depth', 10, 30),

        'min_samples_split': trial.suggest_int('min_samples_split', 2, 10),

        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),

        'max_features': trial.suggest_float('max_features', 0.5, 1.0),

        'random_state': 42,

        'n_jobs': -1

    }

   

    mae_scores = []

    #  5-Fold CV na zbiorze treningowym

    for train_idx, val_idx in kf.split(X_train_final):

        X_tr, X_val = X_train_final.iloc[train_idx], X_train_final.iloc[val_idx]

        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

       

        model = RandomForestRegressor(**params)

        model.fit(X_tr, y_tr)

       

        preds = model.predict(X_val)

        mae_scores.append(mean_absolute_error(y_val, preds))

       

    return np.mean(mae_scores)



# funkcja celu dla XGBoost

def objective_xgb(trial):

    # optymalizacja hiperparametrów XGBoost

    params = {

        'n_estimators': trial.suggest_int('n_estimators', 200, 1000),

        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.15, log=True),

        'max_depth': trial.suggest_int('max_depth', 4, 12),

        'subsample': trial.suggest_float('subsample', 0.6, 1.0),

        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),

        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),

        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),

        'random_state': 42,

        'n_jobs': -1

    }

   

    mae_scores = []

    #  5-Fold CV na zbiorze treningowym

    for train_idx, val_idx in kf.split(X_train_final):

        X_tr, X_val = X_train_final.iloc[train_idx], X_train_final.iloc[val_idx]

        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

       

        model = XGBRegressor(**params)

        model.fit(X_tr, y_tr)

       

        preds = model.predict(X_val)

        mae_scores.append(mean_absolute_error(y_val, preds))

       

    return np.mean(mae_scores)



# Ustawienie liczby prób dla optymalizacji Bayesowskiej

LICZBA_PROB = 50



# optymalizacja Random Forest

print(f"\nUruchamiam optymalizację Bayesowską dla Random Forest ({LICZBA_PROB} iteracji)...")

start_rf = time.time()

study_rf = optuna.create_study(direction='minimize')

study_rf.optimize(objective_rf, n_trials=LICZBA_PROB)

czas_rf = time.time() - start_rf

print(f"RF Tuning zakończony w {czas_rf:.2f} s. Najlepsze walidacyjne MAE: {study_rf.best_value:.2f} PLN")



# optymalizacja XGBoost

print(f"\nUruchamiam optymalizację Bayesowską dla XGBoost ({LICZBA_PROB} iteracji)...")

start_xgb = time.time()

study_xgb = optuna.create_study(direction='minimize')

study_xgb.optimize(objective_xgb, n_trials=LICZBA_PROB)

czas_xgb = time.time() - start_xgb

print(f"XGBoost Tuning zakończony w {czas_xgb:.2f} s. Najlepsze walidacyjne MAE: {study_xgb.best_value:.2f} PLN")



# zapis najlepszych parametrów 

best_params_rf = study_rf.best_params

best_params_xgb = study_xgb.best_params

with open('best_params_rf.json', 'w') as f: json.dump(best_params_rf, f)
with open('best_params_xgb.json', 'w') as f: json.dump(best_params_xgb, f)
print("hiperparametry zapisane do plików JSON na dysku.")


In [ ]:
# ewaluacja i porównanie modeli docelowych

# Wczytanie parametrów
try:
    with open('best_params_rf.json', 'r') as f: loaded_params_rf = json.load(f)
    with open('best_params_xgb.json', 'r') as f: loaded_params_xgb = json.load(f)
except FileNotFoundError:
    loaded_params_rf, loaded_params_xgb = {}, {}

loaded_params_rf.update({'random_state': 42, 'n_jobs': -1})
loaded_params_xgb.update({'random_state': 42, 'n_jobs': -1})

final_models = {
    'Random Forest (Zoptymalizowany)': RandomForestRegressor(**loaded_params_rf),
    'XGBoost (Zoptymalizowany)': XGBRegressor(**loaded_params_xgb)
}

# Zapisywanie nazw cech na potrzeby pliku XAI
joblib.dump(list(X_train_final.columns), 'model_feature_names.pkl')

# Słowniki do przechowywania prognoz i błędów absolutnych do testu Wilcoxona
dict_preds = {}
dict_abs_errors = {}
wyniki_opt_lista = []

# iteracyjny trening i kalkulacja metryk
for nazwa, model in final_models.items():
    
    # CV
    print("  • Uruchamiam 5-krotną walidację krzyżową (CV)...")
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = cross_val_score(model, X_train_final, y_train, cv=kf, scoring='neg_mean_absolute_error', n_jobs=-1)
    cv_mae_mean = -np.mean(cv_scores)
    
    # Trening 
    start_train = time.time()
    model.fit(X_train_final, y_train)
    czas_treningu = time.time() - start_train
    
    # Predykcja
    start_pred = time.time()
    y_pred = model.predict(X_test_final)
    czas_predykcji = time.time() - start_pred
    
    dict_preds[nazwa] = y_pred
    dict_abs_errors[nazwa] = np.abs(y_test - y_pred)
    
    # Metryki Testowe
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)
    pearson_r, _ = pearsonr(y_test, y_pred)
    
    wyniki_opt_lista.append({
        'Model': nazwa,
        'CV MAE (PLN)': round(cv_mae_mean, 2),
        'MAE (PLN)': round(mae, 2),
        'MSE': round(mse, 2),
        'RMSE (PLN)': round(rmse, 2),
        'R2 Score': round(r2, 4),
        'Korelacja Pearsona (r)': round(pearson_r, 4),
        'Czas treningu (s)': round(czas_treningu, 2),
        'Czas predykcji (s)': round(czas_predykcji, 4)
    })

# Raport końcowy
df_wyniki_opt = pd.DataFrame(wyniki_opt_lista)
print("\nOSTATECZNY RAPORT PORÓWNAWCZY MODELI")
display(df_wyniki_opt)

In [ ]:
# przeprowadzenie testu Wilcoxona dla porównania błędów absolutnych obu modeli
print("\nSTATYSTYCZNA WERYFIKACJA ISTOTNOŚCI RÓŻNIC")
errors_rf = dict_abs_errors['Random Forest (Zoptymalizowany)']
errors_xgb = dict_abs_errors['XGBoost (Zoptymalizowany)']

# Wykonanie testu
stat_wilc, p_value = wilcoxon(errors_rf, errors_xgb)

print(f"Wynik testu rang znakowanych Wilcoxona:")
print(f"Statystyka testowa: {stat_wilc:.2f}")
print(f"Wartość p-value: {p_value:.6f}")

ALPHA = 0.05
if p_value < ALPHA:
    print(f"Wynik jest ISTOTNY statystycznie (p < {ALPHA}). Odrzucamy hipotezę zerową.")
    print("Różnice w dokładności predykcyjnej obu modeli nie wynikają z szumu losowego.")
else:
    print(f"Wynik NIE JEST istotny statystycznie (p >= {ALPHA}). Brak podstaw do odrzucenia hipotezy zerowej.")
    print("Obserwowane różnice w błędach mogą być dziełem przypadku.")

In [ ]:
# weryfikacja wizualna wyników predykcji obu modeli poprzez wykresy rozrzutu
print("\nGENEROWANIE WYKRESÓW ROZRZUTU (ACTUAL VS PREDICTED)")
sns.set_theme(style="whitegrid")

modele_nazwy = list(final_models.keys())
kolory = ['orange', 'darkred']

for i, nazwa in enumerate(modele_nazwy):
    fig, ax = plt.subplots(figsize=(8, 7))
    
    y_pred_m = dict_preds[nazwa]
    
    ax.scatter(y_test, y_pred_m, color=kolory[i], alpha=0.3, s=10, label='Obserwacje')
    
    min_val = min(y_test.min(), y_pred_m.min())
    max_val = max(y_test.max(), y_pred_m.max())
    ax.plot([min_val, max_val], [min_val, max_val], color='black', linestyle='--', lw=2, label='Idealne dopasowanie (45°)')
    
    lr = LinearRegression()
    lr.fit(y_test.values.reshape(-1, 1), y_pred_m)
    empiryczna_linia = lr.predict(y_test.values.reshape(-1, 1))
    
    ax.plot(y_test, empiryczna_linia, color='blue', lw=2, 
            label=f'Linia regresji (y = {lr.coef_[0]:.2f}x + {lr.intercept_:.2f})')
    
    ax.set_xlabel('Rzeczywista cena pojazdu (PLN)', fontsize=11)
    ax.set_ylabel('Przewidywana cena pojazdu (PLN)', fontsize=11)
    
    
    ax.legend(loc='upper left')
    ax.grid(True, linestyle='--', alpha=0.5)

    plt.tight_layout()
    plt.show()

In [ ]:
# zapisanie modeli
try:
    joblib.dump(final_models['Random Forest (Zoptymalizowany)'], 'model_rf_optimized.pkl')
    final_models['XGBoost (Zoptymalizowany)'].save_model('model_xgb_optimized.json')
    print("\nObie zoptymalizowane struktury modeli zostały zamrożone na dysku!")
except Exception as e:
    print(f"Wystąpił techniczny problem podczas zapisu plików binarnych: {e}")